# Rugby League Project teamsheet scraper diagnostics

Use this notebook to inspect the actual HTML returned by Rugby League Project before changing the production scraper again.

It deliberately tests one 2023 fixture first, then the existing scraper functions.

In [8]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    parent
    for parent in Path.cwd().resolve().parents
    if (parent / "pyproject.toml").exists()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

/workspaces/rugby_league_pricing_research


In [2]:
from __future__ import annotations

import inspect
import random
import re
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.rugbyleagueproject.org"

SESSION = requests.Session()
SESSION.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/150.0 Safari/537.36"
        )
    }
)


def get_page(
    url: str,
    timeout: int = 30,
    max_retries: int = 5,
    sleep_after_success: bool = False,
) -> requests.Response:
    for attempt in range(max_retries):
        try:
            response = SESSION.get(
                url,
                timeout=timeout,
            )
            response.raise_for_status()

            if sleep_after_success:
                time.sleep(
                    random.uniform(2.0, 4.0)
                )

            return response

        except requests.RequestException:
            if attempt == max_retries - 1:
                raise

            wait_seconds = 10 * (2 ** attempt)

            print(
                f"Request failed for {url}. "
                f"Retrying in {wait_seconds}s..."
            )

            time.sleep(wait_seconds)

## 1. Confirm the season results page

This should return HTTP 200 and show the page title.

In [3]:
season = 2023
season_url = (
    f"{BASE_URL}/seasons/super-league-{season}/results.html"
)

response = get_page(season_url)

print("status:", response.status_code)
print("url:", response.url)
print("bytes:", len(response.content))

soup = BeautifulSoup(
    response.content,
    "html.parser",
)

print("title:", soup.title.get_text(strip=True) if soup.title else None)

status: 200
url: https://www.rugbyleagueproject.org/seasons/super-league-2023/results.html
bytes: 125745
title: 2023 Betfred Super League - Fixtures/Results


## 2. Inspect the results table structure

This prints the first rows with their text and links so we can see exactly how round headings and fixture rows are represented.

In [4]:
content = soup.find(id="content")
match_list = content.find(class_="list") if content else None

print("content found:", content is not None)
print("list found:", match_list is not None)

rows = match_list.find_all("tr") if match_list else []

for i, row in enumerate(rows[:35]):
    print("=" * 100)
    print("ROW", i)
    print("TEXT:", " | ".join(row.stripped_strings))

    for anchor in row.find_all("a", href=True):
        print(
            "LINK:",
            anchor.get_text(" ", strip=True),
            "->",
            anchor["href"],
        )

content found: True
list found: True
ROW 0
TEXT: To view more info on a match, click the | > | button.
ROW 1
TEXT: Round 1
LINK: Round 1 -> /seasons/super-league-2023/round-1/summary.html
ROW 2
TEXT: Feb 16 | Thu 8:00pm | Warrington Wolves | 42 | Leeds Rhinos | 10 | L. Moore | Halliwell Jones Stadium | 11,082 | >
LINK: Warrington Wolves -> /seasons/super-league-2023/warrington-wolves/summary.html
LINK: Leeds Rhinos -> /seasons/super-league-2023/leeds-rhinos/summary.html
LINK: L. Moore -> /referees/42731
LINK: Halliwell Jones Stadium -> /venues/68
LINK: > -> /matches/54214
ROW 3
TEXT: 17 | Fri 7:30pm | Wakefield Trinity | 24 | Catalans Dragons | 38 | T. Grant | Be Well Support Stadium | 4,076 | >
LINK: Wakefield Trinity -> /seasons/super-league-2023/wakefield-trinity/summary.html
LINK: Catalans Dragons -> /seasons/super-league-2023/catalans-dragons/summary.html
LINK: T. Grant -> /referees/42703
LINK: Be Well Support Stadium -> /venues/67
LINK: > -> /matches/54215
ROW 4
TEXT: 17 | Fri 8:

## 3. Test the current season discovery function

This uses your actual `scraper.py`, not duplicated notebook logic.

In [9]:
from scripts.teamsheets.rugby_league_project.scraper import (
    scrape_season_match_references,
)

matches = scrape_season_match_references(2023)

print("matches:", len(matches))

for match in matches[:10]:
    print(match)

matches: 178
MatchReference(season=2023, match_date_text='Feb 16', home_team='Warrington Wolves', away_team='Leeds Rhinos', summary_url='https://www.rugbyleagueproject.org/seasons/super-league-2023/round-1/warrington-wolves-vs-leeds-rhinos/summary.html', source_match_id='super-league-2023_round-1_warrington-wolves-vs-leeds-rhinos')
MatchReference(season=2023, match_date_text='17', home_team='Wakefield Trinity', away_team='Catalans Dragons', summary_url='https://www.rugbyleagueproject.org/seasons/super-league-2023/round-1/wakefield-trinity-vs-catalans-dragons/summary.html', source_match_id='super-league-2023_round-1_wakefield-trinity-vs-catalans-dragons')
MatchReference(season=2023, match_date_text='17', home_team='Leigh Leopards', away_team='Salford Red Devils', summary_url='https://www.rugbyleagueproject.org/seasons/super-league-2023/round-1/leigh-leopards-vs-salford-red-devils/summary.html', source_match_id='super-league-2023_round-1_leigh-leopards-vs-salford-red-devils')
MatchRefere

## 4. Inspect one real match page

Start with Warrington Wolves vs Leeds Rhinos, Round 1.

In [10]:
match_url = (
    "https://www.rugbyleagueproject.org/"
    "seasons/super-league-2023/round-1/"
    "warrington-wolves-vs-leeds-rhinos/summary.html"
)

match_response = get_page(match_url)

print("status:", match_response.status_code)
print("url:", match_response.url)
print("bytes:", len(match_response.content))

match_soup = BeautifulSoup(
    match_response.content,
    "html.parser",
)

print(
    "title:",
    match_soup.title.get_text(strip=True)
    if match_soup.title
    else None,
)

status: 200
url: https://www.rugbyleagueproject.org/seasons/super-league-2023/round-1/warrington-wolves-vs-leeds-rhinos/summary.html
bytes: 50773
title: 2023 Betfred Super League - Round 1 - Warrington Wolves 42 def. Leeds Rhinos 10


## 5. Inspect `.program` and every row

This is the important diagnostic. It shows the exact text, classes, `<th>` values and `name` cells present on the match page.

In [11]:
match_content = match_soup.find(id="content")
program = (
    match_content.find(class_="program")
    if match_content
    else None
)

print("content found:", match_content is not None)
print("program found:", program is not None)

program_rows = program.find_all("tr") if program else []

print("program rows:", len(program_rows))

for i, row in enumerate(program_rows):
    print("=" * 100)
    print("ROW", i)
    print("TEXT:", repr(row.get_text()))
    print("STRIPPED:", " | ".join(row.stripped_strings))
    print("ROW CLASSES:", row.get("class"))

    ths = [
        th.get_text(" ", strip=True)
        for th in row.find_all("th")
    ]
    print("TH:", ths)

    name_cells = row.find_all(class_="name")
    print(
        "NAME CELLS:",
        [
            {
                "text": cell.get_text(" ", strip=True),
                "classes": cell.get("class"),
                "href": (
                    cell.find("a").get("href")
                    if cell.find("a")
                    else None
                ),
            }
            for cell in name_cells
        ],
    )

content found: True
program found: True
program rows: 52
ROW 0
TEXT: '2023 Betfred Super LeagueRound 1'
STRIPPED: 2023 Betfred Super League | Round 1
ROW CLASSES: ['rule_bottom']
TH: ['2023 Betfred Super League Round 1']
NAME CELLS: []
ROW 1
TEXT: 'WarringtonWolves42–10LeedsRhinos'
STRIPPED: Warrington | Wolves | 42 | – | 10 | Leeds | Rhinos
ROW CLASSES: None
TH: ['Warrington Wolves 42', '–', '10 Leeds Rhinos']
NAME CELLS: []
ROW 2
TEXT: 'WarringtonWolves42'
STRIPPED: Warrington | Wolves | 42
ROW CLASSES: None
TH: []
NAME CELLS: []
ROW 3
TEXT: '10LeedsRhinos'
STRIPPED: 10 | Leeds | Rhinos
ROW CLASSES: None
TH: []
NAME CELLS: []
ROW 4
TEXT: 'Match Info'
STRIPPED: Match Info
ROW CLASSES: None
TH: ['Match Info']
NAME CELLS: []
ROW 5
TEXT: 'Match URLwww.rugbyleagueproject.org/matches/54214'
STRIPPED: Match URL | www.rugbyleagueproject.org/matches/54214
ROW CLASSES: ['fname']
TH: ['Match URL']
NAME CELLS: [{'text': 'www.rugbyleagueproject.org/matches/54214', 'classes': ['name', 'pad'], 'hre

## 6. Show rows that look like player rows

This applies the original notebook selectors without requiring a literal `Teams` heading.

In [12]:
candidate_rows = []

for i, row in enumerate(program_rows):
    home_cells = row.find_all(class_="name left")
    name_cells = row.find_all(class_="name")
    th_cells = row.find_all("th")

    if (
        len(home_cells) >= 1
        and len(name_cells) >= 2
        and len(th_cells) >= 1
    ):
        candidate_rows.append(
            {
                "row_index": i,
                "position": th_cells[0].get_text(" ", strip=True),
                "home": home_cells[0].get_text(" ", strip=True),
                "away": name_cells[1].get_text(" ", strip=True),
            }
        )

print("candidate rows:", len(candidate_rows))

for row in candidate_rows:
    print(row)

candidate rows: 25
{'row_index': 18, 'position': 'T', 'home': 'Daryl CLARK', 'away': 'Derrell OLPHERTS'}
{'row_index': 19, 'position': '', 'home': 'Matt DUFTY', 'away': 'Justin SANGARÉ'}
{'row_index': 20, 'position': '', 'home': 'James HARRISON', 'away': ''}
{'row_index': 21, 'position': '', 'home': 'Sam KASIANO', 'away': ''}
{'row_index': 22, 'position': '', 'home': 'Greg MINIKIN', 'away': ''}
{'row_index': 23, 'position': '', 'home': 'Josh THEWLIS', 'away': ''}
{'row_index': 24, 'position': '', 'home': 'Danny WALKER', 'away': ''}
{'row_index': 26, 'position': 'G', 'home': 'Stefan RATCHFORD', 'away': 'Rhyse MARTIN'}
{'row_index': 28, 'position': 'FB', 'home': 'Matt DUFTY', 'away': 'Richard MYLER'}
{'row_index': 29, 'position': 'W', 'home': 'Josh THEWLIS', 'away': 'Luis ROBERTS'}
{'row_index': 30, 'position': 'C', 'home': "Peter MATA'UTIA", 'away': "David FUSITU'A"}
{'row_index': 31, 'position': 'C', 'home': 'Stefan RATCHFORD (c)', 'away': 'Ash HANDLEY'}
{'row_index': 32, 'position': '

## 7. Test the production match teamsheet parser

If this fails, the preceding cells will show exactly why.

In [13]:
from scripts.teamsheets.rugby_league_project.scraper import (
    scrape_match_teamsheet,
)

players = scrape_match_teamsheet(match_url)

print("player records:", len(players))

for player in players:
    print(player)

player records: 34
{'side': 'home', 'player_name': 'Daryl CLARK', 'source_player_id': '16464', 'position': 'T', 'lineup_order': 1, 'is_starting': True}
{'side': 'away', 'player_name': 'Derrell OLPHERTS', 'source_player_id': '19891', 'position': 'T', 'lineup_order': 1, 'is_starting': True}
{'side': 'home', 'player_name': 'Matt DUFTY', 'source_player_id': '23900', 'position': None, 'lineup_order': 2, 'is_starting': True}
{'side': 'away', 'player_name': 'Justin SANGARÉ', 'source_player_id': '23669', 'position': None, 'lineup_order': 2, 'is_starting': True}
{'side': 'home', 'player_name': 'Stefan RATCHFORD', 'source_player_id': '5709', 'position': 'G', 'lineup_order': 3, 'is_starting': True}
{'side': 'away', 'player_name': 'Rhyse MARTIN', 'source_player_id': '21524', 'position': 'G', 'lineup_order': 3, 'is_starting': True}
{'side': 'home', 'player_name': 'Matt DUFTY', 'source_player_id': '23900', 'position': 'FB', 'lineup_order': 4, 'is_starting': True}
{'side': 'away', 'player_name': 'Ric

## 8. Inspect `apply_team_ids` contract

Your latest traceback shows `apply_team_ids()` expects `home_team_name` and `away_team_name`. This prints the live function source so the teamsheet ingestion can use exactly the same contract as fixture ingestion.

In [14]:
from scripts.rugby_league_project.teams_mapping import (
    apply_team_ids,
)

print(inspect.getsource(apply_team_ids))

def apply_team_ids(
    connection: sqlite3.Connection,
    matches: list[dict[str, Any]],
    create_missing: bool = True,
) -> list[dict[str, Any]]:
    mapped_matches: list[dict[str, Any]] = []

    for match in matches:
        season = int(match["season"])

        if create_missing:
            home_team_id = get_or_create_team_id(
                connection=connection,
                source_team_name=match["home_team_name"],
                season=season,
            )
            away_team_id = get_or_create_team_id(
                connection=connection,
                source_team_name=match["away_team_name"],
                season=season,
            )

        else:
            home_team_id = resolve_team_id(
                connection=connection,
                source_team_name=match["home_team_name"],
                season=season,
            )
            away_team_id = resolve_team_id(
                connection=connection,
                source_team_name=match["aw

## 9. Confirm the teamsheet mapping payload

Based on the traceback, the payload passed to `apply_team_ids()` should use the same key names as the fixtures pipeline.

In [15]:
match = matches[0]

mapping_payload = {
    "season": match.season,
    "home_team_name": match.home_team,
    "away_team_name": match.away_team,
    "source_name": "rugby_league_project",
    "source_match_id": match.source_match_id,
}

mapping_payload

{'season': 2023,
 'home_team_name': 'Warrington Wolves',
 'away_team_name': 'Leeds Rhinos',
 'source_name': 'rugby_league_project',
 'source_match_id': 'super-league-2023_round-1_warrington-wolves-vs-leeds-rhinos'}